# MiniMax H3 — ComfyUI Extender on Colab

Long-video test path: **MiniMax H3 + ComfyUI + H3 Extender**.

Start with **2 short connected clips**. Once continuity works, increase the sequence length.

Recommended: **A100 + High RAM**.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Change these before running if needed.
PERSIST_MODELS_TO_DRIVE = False  # H3 models take tens of GB.
PERSIST_OUTPUT_TO_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/MiniMax_H3_ComfyUI"

print("Drive root:", DRIVE_ROOT)
print("Persist models:", PERSIST_MODELS_TO_DRIVE)
print("Persist output/user:", PERSIST_OUTPUT_TO_DRIVE)


## 1. Clone our H3 branch

This cell is safe to rerun: it first moves to `/content` before deleting the previous working copy.


In [ ]:
%cd /content
!rm -rf /content/All-testing /content/minimax_h3_comfy
!git clone --depth 1 --branch minimax-h3-colab https://github.com/Logan17de/All-testing.git /content/All-testing
!cp -r /content/All-testing/video/minimax_h3_comfy /content/minimax_h3_comfy
%cd /content/minimax_h3_comfy
!ls -la


## 2. Install/update ComfyUI + H3 Extender


In [ ]:
import os
os.environ["COMFY_ROOT"] = "/content/ComfyUI"
os.environ["H3_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["H3_PERSIST_MODELS"] = "1" if PERSIST_MODELS_TO_DRIVE else "0"
os.environ["H3_PERSIST_OUTPUT"] = "1" if PERSIST_OUTPUT_TO_DRIVE else "0"

!bash install_comfy_h3.sh


## 3. Download the H3 Ref2VA model set

This is the large download. Re-running the cell skips files that are already present.


In [ ]:
MODEL_ROOT = f"{DRIVE_ROOT}/models" if PERSIST_MODELS_TO_DRIVE else "/content/ComfyUI/models"
!python download_models.py --profile ref2va-int8 --model-root "$MODEL_ROOT" 


## 4. Prepare the current H3 Extender workflow


In [ ]:
!python prepare_workflow.py --comfy-root /content/ComfyUI
!ls -lh /content/ComfyUI/user/default/workflows/


## 5. Launch ComfyUI

The launcher prints a temporary `trycloudflare.com` URL and verifies it. It now forces HTTP/2 because hosted notebook networks can have trouble with the default QUIC transport.

If the public link still fails, **the next cell embeds ComfyUI directly inside Colab**.


In [ ]:
!bash launch_comfy.sh


## 5b. Open ComfyUI directly inside Colab — reliable fallback

Use this whenever the `trycloudflare.com` URL will not open. ComfyUI appears below this cell and stays connected to port 8188 in the current Colab runtime.


In [ ]:
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(
    8188,
    width="100%",
    height="1000",
    cache_in_notebook=False,
)


## 6. Health check / logs

Run this if ComfyUI opens but a node fails.


In [ ]:
import requests, json, pathlib
stats = requests.get("http://127.0.0.1:8188/system_stats", timeout=10).json()
print(json.dumps(stats, indent=2)[:5000])
print("\n--- ComfyUI log tail ---")
print(pathlib.Path("/content/h3_comfy_logs/comfyui.log").read_text(errors="ignore")[-6000:])

print("\n--- Cloudflare log tail ---")
cf = pathlib.Path("/content/h3_comfy_logs/cloudflared.log")
if cf.exists():
    print(cf.read_text(errors="ignore")[-4000:])


## First continuity test

Inside the Extender workflow:

1. Add **one clean character reference**.
2. Use only **2 clips**.
3. Keep the same character/environment definition at the start of both prompts.
4. Generate clip 1 and preview it.
5. If it is good, **Validate** it.
6. Generate clip 2 from the carried motion/audio context.
7. Only after this works, extend to more clips.

The goal of the first run is not a pretty movie — it is proving that **clip 2 continues clip 1 without a visible identity/motion break**.
